# Production Pipeline: Aesthetic Scoring

This notebook is a fully self-contained production pipeline for the aesthetic image scoring system. It benchmarks the recommended configuration for each pipeline component and validates end-to-end performance under realistic production workloads.

**No prior notebooks are required.** All Docker setup, model loading, and benchmarking are included.

## Production Architecture

The app serves multiple concurrent users worldwide. The pipeline has four workloads:

1.  **Onboarding** (one-time per user): CLIP ViT encodes uploaded images → Global MLP scores them all
2.  **Daily new photos**: 1–10 new photos per user, scored immediately via Global MLP
3.  **Weekly personalized re-scoring**: After retraining user embeddings, Personalized MLP re-scores all images for all active users
4.  **New photos between retraining**: Get Global MLP score immediately; personalized scores come at next weekly retrain

**Typical photo library sizes:** - Average user: ~2,000–5,000 photos - Power user: ~10,000–20,000 - Worst case (photography enthusiast): ~50,000

## Production Component Selection

| Component | Best Variant | Batch Size | Rationale |
|------------------|--------------------|------------------|------------------|
| ViT Encoder | Compiled, bs=128 | 128 | Peak throughput (877 FPS) at lowest GPU memory (3.4 GB) |
| Global MLP | ONNX Dynamic INT8 | 32 | 75% smaller model (0.47 MB), identical quality to FP32 |
| Personalized MLP | ONNX Graph-optimized | 32 | Simpler than static quantization (which failed to quantize), same quality |
| Triton Serving | 2 instances + dynamic batching | — | +38% throughput, -49% queue delay at concurrency=8 |

## Prerequisites

This notebook assumes:

1.  You have a Chameleon bare-metal GPU node (A100/A30) with Docker and NVIDIA container toolkit installed (notebooks 0–2 handle provisioning, notebook 2 sets up Docker and NVIDIA toolkit).
2.  The `aesthetic_data` Docker volume exists and is populated (notebook 3 handles data preparation).
3.  The production ONNX models exist in `workspace/models/production/` (included in the git repo — no prior notebooks needed to generate them).

You do **not** need to run notebooks 4–10. This notebook has its own Docker setup that is independent from the `jupyter-onnx-gpu` container used by notebooks 5–10.

## Launch the production environment

If any containers from previous notebooks are still running, stop them first:

``` bash
# runs on node-serve-model
docker stop jupyter triton_server jupyter_triton jupyter_fastapi fastapi_server triton_production jupyter_production 2>/dev/null; true
```

Bring up the production stack (Triton + GPU Jupyter + perf_analyzer SDK):

``` bash
# runs on node-serve-model
docker compose -f ~/aesthetic-hub-serving/docker/docker-compose-production.yaml up -d
```

Wait for Triton to load both models (~30 seconds), then verify:

``` bash
# runs on node-serve-model
docker logs triton_production 2>&1 | tail -10
```

You should see `Started GRPCInferenceService` and `Started HTTPService`, plus both models loaded with 2 instances each.

Get the Jupyter token:

``` bash
# runs on node-serve-model
docker exec jupyter_production jupyter server list
```

Open the Jupyter URL in your browser (replace `localhost` with your floating IP), navigate to `work/`, and open `11_hub_production.ipynb`.

In [36]:
# runs in jupyter container on node-serve-model
import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
import onnxruntime as ort
import psutil
import subprocess
import threading

In [37]:
# runs in jupyter container on node-serve-model
import clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

def normalized(a, axis=-1, order=2):
    l2 = np.atleast_1d(np.linalg.norm(a, order, axis))
    l2[l2 == 0] = 1
    return a / np.expand_dims(l2, axis)

Device: cuda
GPU: NVIDIA A100 80GB PCIe
GPU Memory: 79.3 GB


In [38]:
# runs in jupyter container on node-serve-model
class ResourceMonitor:
    """Polls nvidia-smi (GPU) and psutil (CPU/RAM) in a background thread."""

    def __init__(self, interval=0.5):
        self.interval = interval
        self._stop = threading.Event()
        self.gpu_util = []
        self.gpu_mem_used = []
        self.cpu_percent = []
        self.ram_used_gb = []
        self._thread = None

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                     "--format=csv,noheader,nounits"], text=True
                ).strip().split(",")
                self.gpu_util.append(float(out[0]))
                self.gpu_mem_used.append(float(out[1]))
            except Exception:
                pass
            self.cpu_percent.append(psutil.cpu_percent(interval=None))
            self.ram_used_gb.append(psutil.virtual_memory().used / 1e9)
            time.sleep(self.interval)

    def start(self):
        self._stop.clear()
        self.gpu_util.clear()
        self.gpu_mem_used.clear()
        self.cpu_percent.clear()
        self.ram_used_gb.clear()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread:
            self._thread.join()

    def summary(self, label=""):
        print(f"\nResource usage — {label}")
        if self.gpu_util:
            print(f"  GPU util:  avg={np.mean(self.gpu_util):5.1f}%  peak={max(self.gpu_util):5.1f}%")
            print(f"  GPU mem:   avg={np.mean(self.gpu_mem_used):6.0f} MB  peak={max(self.gpu_mem_used):6.0f} MB")
        print(f"  CPU util:  avg={np.mean(self.cpu_percent):5.1f}%  peak={max(self.cpu_percent):5.1f}%")
        print(f"  RAM used:  avg={np.mean(self.ram_used_gb):5.2f} GB  peak={max(self.ram_used_gb):5.2f} GB")

monitor = ResourceMonitor()
print("ResourceMonitor ready.")

ResourceMonitor ready.


## Prepare test data

In [39]:
# runs in jupyter container on node-serve-model
data_dir = os.getenv("AESTHETIC_DATA_DIR", "flickr-aes")
clip_model, clip_preprocess = clip.load("ViT-L/14", device=device)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'inference'), transform=clip_preprocess)
print(f"Test dataset: {len(test_dataset)} images")

Test dataset: 7000 images


------------------------------------------------------------------------

## Part 1: CLIP ViT-L/14 — Compiled, Batch Size 128

The ViT encoder is the heaviest component in the pipeline. From our benchmarks (notebook 5), the compiled model at batch_size=128 achieves peak throughput (877 FPS) at the lowest GPU memory footprint (3.4 GB). Going to batch_size=256 gains nothing but +900 MB; batch_size=512 actually drops throughput.

**Production use case:** Pre-compute embeddings once per image at upload time. For a typical user onboarding with ~5,000 photos: 5,000 / 128 = 39 batches, wall time ≈ 5.7 seconds.

In [40]:
# runs in jupyter container on node-serve-model
# Compile the ViT visual encoder
clip_model.visual = torch.compile(clip_model.visual)

# Warm-up (first call triggers compilation)
print("Compiling ViT visual encoder...")
loader_warmup = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)
warmup_images, _ = next(iter(loader_warmup))
with torch.no_grad():
    clip_model.encode_image(warmup_images.to(device))
if device.type == "cuda":
    torch.cuda.synchronize()
print("Compilation complete.")

Compiling ViT visual encoder...
Compilation complete.


In [41]:
# runs in jupyter container on node-serve-model
# Benchmark: ViT compiled, batch_size=128
BATCH_SIZE = 128
NUM_BATCHES = 50

loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
images, _ = next(iter(loader))
images = images.to(device)

# Warm-up
with torch.no_grad():
    clip_model.encode_image(images)
if device.type == "cuda":
    torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()
monitor.start()
vit_times = []
with torch.no_grad():
    for _ in range(NUM_BATCHES):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        clip_model.encode_image(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        vit_times.append(time.time() - start)
monitor.stop()
peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6

vit_fps = (BATCH_SIZE * NUM_BATCHES) / np.sum(vit_times)
print(f"ViT-L/14 Compiled — batch_size={BATCH_SIZE}")
print(f"  Median latency:  {np.percentile(vit_times, 50) * 1000:.2f} ms")
print(f"  95th percentile: {np.percentile(vit_times, 95) * 1000:.2f} ms")
print(f"  Throughput:      {vit_fps:.1f} FPS")
print(f"  GPU mem peak:    {peak_mem_mb:.0f} MB")
monitor.summary("ViT compiled bs=128 (production)")

ViT-L/14 Compiled — batch_size=128
  Median latency:  139.36 ms
  95th percentile: 140.42 ms
  Throughput:      918.5 FPS
  GPU mem peak:    1939 MB

Resource usage — ViT compiled bs=128 (production)
  GPU util:  avg= 92.9%  peak=100.0%
  GPU mem:   avg=  3923 MB  peak=  3923 MB
  CPU util:  avg=  0.4%  peak=  0.6%
  RAM used:  avg= 7.52 GB  peak= 7.58 GB


### Production sizing estimate

In [42]:
# runs in jupyter container on node-serve-model
# Production sizing for typical user libraries
for lib_size in [2000, 5000, 10000, 20000, 50000]:
    n_batches = (lib_size + BATCH_SIZE - 1) // BATCH_SIZE
    wall_time = lib_size / vit_fps
    print(f"  {lib_size:>6,} images → {n_batches:>4} batches → {wall_time:.1f}s wall time")

   2,000 images →   16 batches → 2.2s wall time
   5,000 images →   40 batches → 5.4s wall time
  10,000 images →   79 batches → 10.9s wall time
  20,000 images →  157 batches → 21.8s wall time
  50,000 images →  391 batches → 54.4s wall time


------------------------------------------------------------------------

## Part 2: Global MLP — Dynamic INT8 Quantized (CPU)

The Global MLP scores every image with a universal aesthetic quality score. We use the dynamic INT8 quantized ONNX model — 75% smaller than FP32 (0.47 MB vs 1.86 MB) with identical quality (MAE=0.0729, SRCC=0.7731).

**Production use case:** Runs once at onboarding to score all uploaded images, then on each new photo upload. At ~395K FPS on CPU, scoring 5,000 images takes ~0.01 seconds.

In [43]:
# runs in jupyter container on node-serve-model
# Pre-compute CLIP embeddings for MLP benchmarking
print("Pre-computing CLIP embeddings for MLP benchmarks...")
with torch.no_grad():
    emb_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)
    emb_images, _ = next(iter(emb_loader))
    emb_features = clip_model.encode_image(emb_images.to(device))
    all_embeddings = normalized(emb_features.cpu().numpy()).astype(np.float32)
    single_embedding = all_embeddings[:1]
    batch_32_embeddings = all_embeddings[:32]
print(f"Embeddings ready: {all_embeddings.shape}")

Pre-computing CLIP embeddings for MLP benchmarks...
Embeddings ready: (128, 768)


In [44]:
# runs in jupyter container on node-serve-model
# Load production Global MLP (dynamic INT8 quantized)
global_model_path = "models/production/flickr_global_quantized_dynamic.onnx"
global_model_size = os.path.getsize(global_model_path)
print(f"Global MLP model size: {global_model_size / 1e6:.2f} MB")

global_session = ort.InferenceSession(global_model_path, providers=['CPUExecutionProvider'])
print(f"Execution provider: {global_session.get_providers()}")

Global MLP model size: 0.47 MB
Execution provider: ['CPUExecutionProvider']


In [45]:
# runs in jupyter container on node-serve-model
# Sample predictions — verify model produces reasonable scores
outputs = global_session.run(None, {global_session.get_inputs()[0].name: batch_32_embeddings})[0]
scores = outputs.flatten()
print(f"Sample scores (first 5): {', '.join(f'{s:.3f}' for s in scores[:5])}")
print(f"Batch mean: {scores.mean():.3f}, std: {scores.std():.3f}")

Sample scores (first 5): 0.621, 0.500, 0.567, 0.573, 0.543
Batch mean: 0.543, std: 0.087


In [46]:
# runs in jupyter container on node-serve-model
# Benchmark: Global MLP, batch_size=32 (production batch size)
GLOBAL_BATCH_SIZE = 32
NUM_TRIALS_SINGLE = 100
NUM_BATCHES_BATCH = 50

# Single sample latency
global_session.run(None, {global_session.get_inputs()[0].name: single_embedding})  # warm-up

global_single_latencies = []
for _ in range(NUM_TRIALS_SINGLE):
    start = time.time()
    global_session.run(None, {global_session.get_inputs()[0].name: single_embedding})
    global_single_latencies.append(time.time() - start)

# Batch throughput
batch_emb = batch_32_embeddings
global_session.run(None, {global_session.get_inputs()[0].name: batch_emb})  # warm-up

global_batch_times = []
for _ in range(NUM_BATCHES_BATCH):
    start = time.time()
    global_session.run(None, {global_session.get_inputs()[0].name: batch_emb})
    global_batch_times.append(time.time() - start)

global_batch_fps = (GLOBAL_BATCH_SIZE * NUM_BATCHES_BATCH) / np.sum(global_batch_times)
global_single_fps = NUM_TRIALS_SINGLE / np.sum(global_single_latencies)

print(f"Global MLP — Dynamic INT8 (CPU)")
print(f"  Model size:           {global_model_size / 1e6:.2f} MB")
print(f"  Single latency (p50): {np.percentile(global_single_latencies, 50) * 1000:.4f} ms")
print(f"  Single latency (p95): {np.percentile(global_single_latencies, 95) * 1000:.4f} ms")
print(f"  Single throughput:    {global_single_fps:.0f} FPS")
print(f"  Batch throughput (b={GLOBAL_BATCH_SIZE}): {global_batch_fps:.0f} FPS")

Global MLP — Dynamic INT8 (CPU)
  Model size:           0.47 MB
  Single latency (p50): 0.0786 ms
  Single latency (p95): 0.1107 ms
  Single throughput:    11800 FPS
  Batch throughput (b=32): 111393 FPS


In [47]:
# runs in jupyter container on node-serve-model
# Production sizing for Global MLP
for lib_size in [2000, 5000, 10000, 20000, 50000]:
    wall_time = lib_size / global_batch_fps
    print(f"  {lib_size:>6,} images → {wall_time:.4f}s ({wall_time*1000:.1f} ms)")

   2,000 images → 0.0180s (18.0 ms)
   5,000 images → 0.0449s (44.9 ms)
  10,000 images → 0.0898s (89.8 ms)
  20,000 images → 0.1795s (179.5 ms)
  50,000 images → 0.4489s (448.9 ms)


------------------------------------------------------------------------

## Part 3: Personalized MLP — Graph-Optimized (CPU)

The Personalized MLP re-scores all images for each user after weekly retraining. It takes a 768-dim CLIP embedding + a user index and outputs a per-user aesthetic score. We use the graph-optimized ONNX model (1.90 MB, SRCC=0.5920, MAE=0.1721).

**Production use case:** Weekly batch job re-scores all images for all active users. 1,000 users × 5,000 images = 5M images. At ~293K FPS: ~17 seconds total.

In [48]:
# runs in jupyter container on node-serve-model
# Load production Personalized MLP (graph-optimized)
personal_model_path = "models/production/flickr_personalized_optimized.onnx"
personal_model_size = os.path.getsize(personal_model_path)
print(f"Personalized MLP model size: {personal_model_size / 1e6:.2f} MB")

personal_session = ort.InferenceSession(personal_model_path, providers=['CPUExecutionProvider'])
input_names = [i.name for i in personal_session.get_inputs()]
print(f"Execution provider: {personal_session.get_providers()}")
print(f"Inputs: {[(i.name, i.shape, i.type) for i in personal_session.get_inputs()]}")

Personalized MLP model size: 1.90 MB
Execution provider: ['CPUExecutionProvider']
Inputs: [('embedding', ['batch_size', 768], 'tensor(float)'), ('user_idx', ['batch_size'], 'tensor(int64)')]


In [49]:
# runs in jupyter container on node-serve-model
# Sample predictions — verify personalized model
p_user_idx_batch = np.zeros(32, dtype=np.int64)
p_user_idx_single = np.array([0], dtype=np.int64)

outputs = personal_session.run(None, {input_names[0]: batch_32_embeddings, input_names[1]: p_user_idx_batch})[0]
scores = outputs.flatten()
print(f"Sample scores (first 5): {', '.join(f'{s:.3f}' for s in scores[:5])}")
print(f"Batch mean: {scores.mean():.3f}, std: {scores.std():.3f}")

Sample scores (first 5): 0.587, 0.263, 0.397, 0.423, 0.419
Batch mean: 0.398, std: 0.118


In [50]:
# runs in jupyter container on node-serve-model
# Benchmark: Personalized MLP, batch_size=32
PERSONAL_BATCH_SIZE = 32

# Single sample latency
personal_session.run(None, {input_names[0]: single_embedding, input_names[1]: p_user_idx_single})  # warm-up

personal_single_latencies = []
for _ in range(NUM_TRIALS_SINGLE):
    start = time.time()
    personal_session.run(None, {input_names[0]: single_embedding, input_names[1]: p_user_idx_single})
    personal_single_latencies.append(time.time() - start)

# Batch throughput
personal_session.run(None, {input_names[0]: batch_32_embeddings, input_names[1]: p_user_idx_batch})  # warm-up

personal_batch_times = []
for _ in range(NUM_BATCHES_BATCH):
    start = time.time()
    personal_session.run(None, {input_names[0]: batch_32_embeddings, input_names[1]: p_user_idx_batch})
    personal_batch_times.append(time.time() - start)

personal_batch_fps = (PERSONAL_BATCH_SIZE * NUM_BATCHES_BATCH) / np.sum(personal_batch_times)
personal_single_fps = NUM_TRIALS_SINGLE / np.sum(personal_single_latencies)

print(f"Personalized MLP — Graph-Optimized (CPU)")
print(f"  Model size:           {personal_model_size / 1e6:.2f} MB")
print(f"  Single latency (p50): {np.percentile(personal_single_latencies, 50) * 1000:.4f} ms")
print(f"  Single latency (p95): {np.percentile(personal_single_latencies, 95) * 1000:.4f} ms")
print(f"  Single throughput:    {personal_single_fps:.0f} FPS")
print(f"  Batch throughput (b={PERSONAL_BATCH_SIZE}): {personal_batch_fps:.0f} FPS")

Personalized MLP — Graph-Optimized (CPU)
  Model size:           1.90 MB
  Single latency (p50): 0.0699 ms
  Single latency (p95): 0.0860 ms
  Single throughput:    12522 FPS
  Batch throughput (b=32): 81902 FPS


In [51]:
# runs in jupyter container on node-serve-model
# Production sizing for weekly re-scoring
for n_users in [100, 500, 1000, 5000]:
    total_images = n_users * 5000
    wall_time = total_images / personal_batch_fps
    print(f"  {n_users:>5,} users × 5,000 imgs = {total_images:>10,} images → {wall_time:.1f}s")

    100 users × 5,000 imgs =    500,000 images → 6.1s
    500 users × 5,000 imgs =  2,500,000 images → 30.5s
  1,000 users × 5,000 imgs =  5,000,000 images → 61.0s
  5,000 users × 5,000 imgs = 25,000,000 images → 305.2s


------------------------------------------------------------------------

## Part 4: Triton Inference Server — Production Configuration

Now we benchmark the MLP heads served via NVIDIA Triton with the production configuration:

- **2 model instances** per model on GPU 0 — our NB10 data showed +38% throughput and -49% queue delay at concurrency=8
- **Dynamic batching** with `preferred_batch_size: [4, 8, 16, 32, 64, 128]` and `max_queue_delay: 50 µs` — groups concurrent single-image requests from multiple users into efficient batches
- **max_batch_size: 128** — matches our ViT batch size for consistency
- **Always-on server** — users in different time zones means no good maintenance window

The production docker-compose has already started Triton with these settings. Let’s verify and benchmark.

In [52]:
# runs in jupyter container on node-serve-model
import tritonclient.http as httpclient

### Verify Triton is ready with production configs

In [53]:
# runs in jupyter container on node-serve-model
TRITON_URL = "triton_server:8000"
client = httpclient.InferenceServerClient(url=TRITON_URL)

print(f"Server live:  {client.is_server_live()}")
print(f"Server ready: {client.is_server_ready()}")
print()

for model_name in ["flickr_global", "flickr_personalized"]:
    ready = client.is_model_ready(model_name)
    meta = client.get_model_metadata(model_name)
    config = client.get_model_config(model_name)
    print(f"Model '{model_name}': ready={ready}")
    for inp in meta['inputs']:
        print(f"  Input:  {inp['name']:>15s}  shape={inp['shape']}  dtype={inp['datatype']}")
    for out in meta['outputs']:
        print(f"  Output: {out['name']:>15s}  shape={out['shape']}  dtype={out['datatype']}")
    print()

Server live:  True
Server ready: True

Model 'flickr_global': ready=True
  Input:            input  shape=[-1, 768]  dtype=FP32
  Output:          output  shape=[-1, 1]  dtype=FP32

Model 'flickr_personalized': ready=True
  Input:        embedding  shape=[-1, 768]  dtype=FP32
  Input:         user_idx  shape=[-1, 1]  dtype=INT64
  Output:          output  shape=[-1, 1]  dtype=FP32



### Triton endpoint URLs

In [54]:
# runs in jupyter container on node-serve-model
BASE = f"http://{TRITON_URL}"
print("Triton HTTP API endpoints:")
print(f"  Health:                {BASE}/v2/health/ready")
print(f"  Server metadata:       {BASE}/v2")
print()
for model_name in ["flickr_global", "flickr_personalized"]:
    print(f"  [{model_name}]")
    print(f"    Metadata:          {BASE}/v2/models/{model_name}")
    print(f"    Inference:         {BASE}/v2/models/{model_name}/infer")
    print(f"    Stats:             {BASE}/v2/models/{model_name}/versions/1/stats")
    print()
print(f"  Prometheus metrics:    http://triton_server:8002/metrics")

Triton HTTP API endpoints:
  Health:                http://triton_server:8000/v2/health/ready
  Server metadata:       http://triton_server:8000/v2

  [flickr_global]
    Metadata:          http://triton_server:8000/v2/models/flickr_global
    Inference:         http://triton_server:8000/v2/models/flickr_global/infer
    Stats:             http://triton_server:8000/v2/models/flickr_global/versions/1/stats

  [flickr_personalized]
    Metadata:          http://triton_server:8000/v2/models/flickr_personalized
    Inference:         http://triton_server:8000/v2/models/flickr_personalized/infer
    Stats:             http://triton_server:8000/v2/models/flickr_personalized/versions/1/stats

  Prometheus metrics:    http://triton_server:8002/metrics


### Server-side GPU metrics via Triton Prometheus endpoint

In [55]:
# runs in jupyter container on node-serve-model
import requests as _req

def triton_gpu_stats():
    """Fetch GPU utilization and memory from Triton's Prometheus metrics endpoint."""
    try:
        lines = _req.get("http://triton_server:8002/metrics", timeout=2).text.splitlines()
    except Exception as e:
        print(f"Could not reach Triton metrics endpoint: {e}")
        return
    keys = {
        "nv_gpu_utilization": "GPU utilization (%)",
        "nv_gpu_memory_used_bytes": "GPU memory used (MB)",
        "nv_gpu_memory_total_bytes": "GPU memory total (MB)",
        "nv_inference_count": "Total inferences served",
    }
    print("Triton server-side GPU metrics:")
    for line in lines:
        for key, label in keys.items():
            if line.startswith(key) and not line.startswith("#"):
                val = float(line.split()[-1])
                if "bytes" in key:
                    val /= 1024 ** 2
                print(f"  {label}: {val:.1f}")

triton_gpu_stats()

Triton server-side GPU metrics:
  Total inferences served: 5624653.0
  Total inferences served: 6032717.0
  GPU utilization (%): 0.0
  GPU memory total (MB): 81920.0
  GPU memory used (MB): 3922.0


### Prepare test data for Triton benchmarks

In [56]:
# runs in jupyter container on node-serve-model
rng = np.random.default_rng(42)

# Single sample (simulates daily new photo)
single_emb = rng.standard_normal(768).astype(np.float32)
single_emb = (single_emb / np.linalg.norm(single_emb)).reshape(1, 768)

# Batch of 32 (simulates moderate traffic)
batch_32 = rng.standard_normal((32, 768)).astype(np.float32)
batch_32 = batch_32 / np.linalg.norm(batch_32, axis=1, keepdims=True)

# Batch of 64 (simulates onboarding batch)
batch_64 = rng.standard_normal((64, 768)).astype(np.float32)
batch_64 = batch_64 / np.linalg.norm(batch_64, axis=1, keepdims=True)

# User indices
single_user_idx = np.array([[0]], dtype=np.int64)
batch_32_user_idx = np.zeros((32, 1), dtype=np.int64)
batch_64_user_idx = np.zeros((64, 1), dtype=np.int64)

print(f"Test data ready: single={single_emb.shape}, batch_32={batch_32.shape}, batch_64={batch_64.shape}")

Test data ready: single=(1, 768), batch_32=(32, 768), batch_64=(64, 768)


### Helper functions

In [57]:
# runs in jupyter container on node-serve-model
def infer_global(client, embeddings):
    """Send a global model inference request to Triton."""
    inputs = [httpclient.InferInput("input", embeddings.shape, "FP32")]
    inputs[0].set_data_from_numpy(embeddings)
    outputs = [httpclient.InferRequestedOutput("output")]
    result = client.infer(model_name="flickr_global", inputs=inputs, outputs=outputs)
    return result.as_numpy("output")

def infer_personalized(client, embeddings, user_indices):
    """Send a personalized model inference request to Triton."""
    inp_emb = httpclient.InferInput("embedding", embeddings.shape, "FP32")
    inp_emb.set_data_from_numpy(embeddings)
    inp_idx = httpclient.InferInput("user_idx", user_indices.shape, "INT64")
    inp_idx.set_data_from_numpy(user_indices)
    outputs = [httpclient.InferRequestedOutput("output")]
    result = client.infer(model_name="flickr_personalized", inputs=[inp_emb, inp_idx], outputs=outputs)
    return result.as_numpy("output")

### Sanity check — both models

In [58]:
# runs in jupyter container on node-serve-model
# Global MLP
scores = infer_global(client, single_emb)
print(f"Global single:  {scores.flatten()[0]:.4f}")
scores = infer_global(client, batch_32)
print(f"Global batch=32 (first 5): {', '.join(f'{s:.4f}' for s in scores.flatten()[:5])}")

# Personalized MLP
scores = infer_personalized(client, single_emb, single_user_idx)
print(f"Personal single: {scores.flatten()[0]:.4f}")
scores = infer_personalized(client, batch_32, batch_32_user_idx)
print(f"Personal batch=32 (first 5): {', '.join(f'{s:.4f}' for s in scores.flatten()[:5])}")

Global single:  0.6003
Global batch=32 (first 5): 0.6602, 0.5495, 0.7445, 0.7462, 0.5640
Personal single: 0.2831
Personal batch=32 (first 5): 0.2816, 0.2355, 0.5396, 0.6310, 0.2885


------------------------------------------------------------------------

### Global MLP — Triton Production Benchmark

In [59]:
# runs in jupyter container on node-serve-model
# Sequential single-sample latency (simulates 1 user adding a photo)
num_trials = 500

for _ in range(20):
    infer_global(client, single_emb)  # warm-up

monitor.start()
global_triton_single = []
for _ in range(num_trials):
    start = time.time()
    infer_global(client, single_emb)
    global_triton_single.append(time.time() - start)
monitor.stop()

global_triton_single = np.array(global_triton_single)
print(f"Global MLP — Triton Single Sample (n={num_trials})")
print(f"  Median latency:  {np.median(global_triton_single)*1000:.2f} ms")
print(f"  95th percentile: {np.percentile(global_triton_single, 95)*1000:.2f} ms")
print(f"  99th percentile: {np.percentile(global_triton_single, 99)*1000:.2f} ms")
print(f"  Throughput:      {num_trials / global_triton_single.sum():.0f} infer/s")
monitor.summary("Global Triton single (production)")
triton_gpu_stats()

Global MLP — Triton Single Sample (n=500)
  Median latency:  0.92 ms
  95th percentile: 1.11 ms
  99th percentile: 1.33 ms
  Throughput:      1058 infer/s

Resource usage — Global Triton single (production)
  GPU util:  avg=  0.0%  peak=  0.0%
  GPU mem:   avg=  3923 MB  peak=  3923 MB
  CPU util:  avg=  5.0%  peak=  5.0%
  RAM used:  avg= 7.11 GB  peak= 7.11 GB
Triton server-side GPU metrics:
  Total inferences served: 5624686.0
  Total inferences served: 6033270.0
  GPU utilization (%): 0.0
  GPU memory total (MB): 81920.0
  GPU memory used (MB): 3922.0


In [60]:
# runs in jupyter container on node-serve-model
# Batch throughput — batch_size=32 (moderate traffic)
num_trials = 200

for _ in range(20):
    infer_global(client, batch_32)

global_triton_b32 = []
for _ in range(num_trials):
    start = time.time()
    infer_global(client, batch_32)
    global_triton_b32.append(time.time() - start)

global_triton_b32 = np.array(global_triton_b32)
g_b32_throughput = (num_trials * 32) / global_triton_b32.sum()
print(f"Global MLP — Triton Batch=32 (n={num_trials})")
print(f"  Median latency:  {np.median(global_triton_b32)*1000:.2f} ms")
print(f"  Throughput:      {g_b32_throughput:.0f} samples/s")

Global MLP — Triton Batch=32 (n=200)
  Median latency:  1.30 ms
  Throughput:      24198 samples/s


In [61]:
# runs in jupyter container on node-serve-model
# Batch throughput — batch_size=64 (onboarding scenario)
num_trials = 200

for _ in range(20):
    infer_global(client, batch_64)

global_triton_b64 = []
for _ in range(num_trials):
    start = time.time()
    infer_global(client, batch_64)
    global_triton_b64.append(time.time() - start)

global_triton_b64 = np.array(global_triton_b64)
g_b64_throughput = (num_trials * 64) / global_triton_b64.sum()
print(f"Global MLP — Triton Batch=64 (n={num_trials})")
print(f"  Median latency:  {np.median(global_triton_b64)*1000:.2f} ms")
print(f"  Throughput:      {g_b64_throughput:.0f} samples/s")

Global MLP — Triton Batch=64 (n=200)
  Median latency:  1.72 ms
  Throughput:      38776 samples/s


------------------------------------------------------------------------

### Personalized MLP — Triton Production Benchmark

In [62]:
# runs in jupyter container on node-serve-model
# Sequential single-sample latency
num_trials = 500

for _ in range(20):
    infer_personalized(client, single_emb, single_user_idx)

personal_triton_single = []
for _ in range(num_trials):
    start = time.time()
    infer_personalized(client, single_emb, single_user_idx)
    personal_triton_single.append(time.time() - start)

personal_triton_single = np.array(personal_triton_single)
print(f"Personalized MLP — Triton Single Sample (n={num_trials})")
print(f"  Median latency:  {np.median(personal_triton_single)*1000:.2f} ms")
print(f"  95th percentile: {np.percentile(personal_triton_single, 95)*1000:.2f} ms")
print(f"  99th percentile: {np.percentile(personal_triton_single, 99)*1000:.2f} ms")
print(f"  Throughput:      {num_trials / personal_triton_single.sum():.0f} infer/s")

Personalized MLP — Triton Single Sample (n=500)
  Median latency:  0.84 ms
  95th percentile: 0.91 ms
  99th percentile: 0.99 ms
  Throughput:      1176 infer/s


In [63]:
# runs in jupyter container on node-serve-model
# Batch throughput — batch_size=32
num_trials = 200

for _ in range(20):
    infer_personalized(client, batch_32, batch_32_user_idx)

personal_triton_b32 = []
for _ in range(num_trials):
    start = time.time()
    infer_personalized(client, batch_32, batch_32_user_idx)
    personal_triton_b32.append(time.time() - start)

personal_triton_b32 = np.array(personal_triton_b32)
p_b32_throughput = (num_trials * 32) / personal_triton_b32.sum()
print(f"Personalized MLP — Triton Batch=32 (n={num_trials})")
print(f"  Median latency:  {np.median(personal_triton_b32)*1000:.2f} ms")
print(f"  Throughput:      {p_b32_throughput:.0f} samples/s")

Personalized MLP — Triton Batch=32 (n=200)
  Median latency:  1.24 ms
  Throughput:      25504 samples/s


In [64]:
# runs in jupyter container on node-serve-model
# Batch throughput — batch_size=64 (weekly re-scoring)
num_trials = 200

for _ in range(20):
    infer_personalized(client, batch_64, batch_64_user_idx)

personal_triton_b64 = []
for _ in range(num_trials):
    start = time.time()
    infer_personalized(client, batch_64, batch_64_user_idx)
    personal_triton_b64.append(time.time() - start)

personal_triton_b64 = np.array(personal_triton_b64)
p_b64_throughput = (num_trials * 64) / personal_triton_b64.sum()
print(f"Personalized MLP — Triton Batch=64 (n={num_trials})")
print(f"  Median latency:  {np.median(personal_triton_b64)*1000:.2f} ms")
print(f"  Throughput:      {p_b64_throughput:.0f} samples/s")

Personalized MLP — Triton Batch=64 (n=200)
  Median latency:  1.78 ms
  Throughput:      35913 samples/s


------------------------------------------------------------------------

### `perf_analyzer` — Concurrency and Batch Sweeps

`perf_analyzer` is Triton’s official load-testing CLI. It runs from the `triton_production_sdk` sidecar container and provides precise latency breakdowns (queue time, compute time) and throughput.

Run these commands on the **host** (SSH into `node-serve-model`):

#### Global MLP — Concurrency sweep (many users, single photo upload)

``` bash
# Concurrency = 1 (single user)
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 1 --shape input:768 --concurrency-range 1

# Concurrency = 8 (multiple users onboarding)
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 1 --shape input:768 --concurrency-range 8

# Concurrency = 16 (peak traffic)
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 1 --shape input:768 --concurrency-range 16
```

#### Global MLP — Batch size sweep (one user, multiple photo uploads)

``` bash
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 1 --shape input:768 --concurrency-range 1 --measurement-interval 20000 --percentile=95
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 32 --shape input:768 --concurrency-range 1 --measurement-interval 10000
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_global -b 64 --shape input:768 --concurrency-range 1 --measurement-interval 10000
```

#### Personalized MLP — Concurrency sweep (many users, single photo updates)

``` bash
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 1
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 8
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 16
```

#### Personalized MLP — Batch size sweep (one user, multiple photo updates)

``` bash
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 1  --shape embedding:768 --shape user_idx:1 --concurrency-range 1 --measurement-interval 10000
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 32 --shape embedding:768 --shape user_idx:1 --concurrency-range 1 --measurement-interval 10000
docker exec triton_production_sdk perf_analyzer -u localhost:8000 -m flickr_personalized -b 64 --shape embedding:768 --shape user_idx:1 --concurrency-range 1 --measurement-interval 10000
```

### Check dynamic batching statistics

After running the benchmarks above, check how Triton formed batches:

``` bash
curl -s http://localhost:8000/v2/models/flickr_global/versions/1/stats | python3 -m json.tool
curl -s http://localhost:8000/v2/models/flickr_personalized/versions/1/stats | python3 -m json.tool
```

In [65]:
# runs in jupyter container on node-serve-model
# Fetch batch stats programmatically
import json

for model_name in ["flickr_global", "flickr_personalized"]:
    try:
        resp = _req.get(f"http://triton_server:8000/v2/models/{model_name}/versions/1/stats", timeout=5)
        stats = resp.json()
        infer_stats = stats.get("model_stats", [{}])[0].get("inference_stats", {})
        batch_stats = stats.get("model_stats", [{}])[0].get("batch_stats", [])
        print(f"\n{model_name}:")
        print(f"  Total inferences: {infer_stats.get('success', {}).get('count', 'N/A')}")
        if batch_stats:
            print(f"  Batch size distribution:")
            for bs in batch_stats:
                count = bs.get("compute_infer", {}).get("count", 0)
                print(f"    batch_size={bs.get('batch_size', '?'):>3}: {count:>8} batches executed")
    except Exception as e:
        print(f"  Could not fetch stats for {model_name}: {e}")


flickr_global:
  Total inferences: 1757344
  Batch size distribution:
    batch_size=  1:   402707 batches executed
    batch_size=  2:    50256 batches executed
    batch_size=  3:    61401 batches executed
    batch_size=  4:   100103 batches executed
    batch_size=  5:    19287 batches executed
    batch_size=  6:    11298 batches executed
    batch_size=  7:     9370 batches executed
    batch_size=  8:    17652 batches executed
    batch_size=  9:     4834 batches executed
    batch_size= 10:     1877 batches executed
    batch_size= 11:      926 batches executed
    batch_size= 12:      824 batches executed
    batch_size= 13:      131 batches executed
    batch_size= 14:        2 batches executed
    batch_size= 15:        1 batches executed
    batch_size= 32:   135846 batches executed
    batch_size= 64:    78539 batches executed

flickr_personalized:
  Total inferences: 2437393
  Batch size distribution:
    batch_size=  1:   332645 batches executed
    batch_size=  2:    9

------------------------------------------------------------------------

## Production Pipeline Summary

In [66]:
# runs in jupyter container on node-serve-model
print("=" * 80)
print("PRODUCTION PIPELINE BENCHMARK SUMMARY")
print("=" * 80)

print(f"\n{'Component':<35} {'Variant':<25} {'Metric':<20} {'Value':>10}")
print("-" * 90)

# ViT
print(f"{'ViT-L/14':<35} {'Compiled, bs=128':<25} {'Throughput (FPS)':<20} {vit_fps:>10.1f}")
print(f"{'':<35} {'':<25} {'Latency p50 (ms)':<20} {np.percentile(vit_times, 50)*1000:>10.2f}")
print(f"{'':<35} {'':<25} {'GPU mem (MB)':<20} {peak_mem_mb:>10.0f}")

# Global MLP (CPU)
print(f"{'Global MLP (CPU)':<35} {'Dynamic INT8':<25} {'Batch FPS (b=32)':<20} {global_batch_fps:>10.0f}")
print(f"{'':<35} {'':<25} {'Single FPS':<20} {global_single_fps:>10.0f}")
print(f"{'':<35} {'':<25} {'Model size (MB)':<20} {global_model_size/1e6:>10.2f}")

# Personalized MLP (CPU)
print(f"{'Personalized MLP (CPU)':<35} {'Graph-optimized':<25} {'Batch FPS (b=32)':<20} {personal_batch_fps:>10.0f}")
print(f"{'':<35} {'':<25} {'Single FPS':<20} {personal_single_fps:>10.0f}")
print(f"{'':<35} {'':<25} {'Model size (MB)':<20} {personal_model_size/1e6:>10.2f}")

# Triton Global
print(f"{'Global MLP (Triton)':<35} {'2 inst + dyn batch':<25} {'Single (infer/s)':<20} {num_trials/global_triton_single.sum():>10.0f}")
print(f"{'':<35} {'':<25} {'Batch=32 (samp/s)':<20} {g_b32_throughput:>10.0f}")
print(f"{'':<35} {'':<25} {'Batch=64 (samp/s)':<20} {g_b64_throughput:>10.0f}")

# Triton Personalized
print(f"{'Personalized MLP (Triton)':<35} {'2 inst + dyn batch':<25} {'Single (infer/s)':<20} {num_trials/personal_triton_single.sum():>10.0f}")
print(f"{'':<35} {'':<25} {'Batch=32 (samp/s)':<20} {p_b32_throughput:>10.0f}")
print(f"{'':<35} {'':<25} {'Batch=64 (samp/s)':<20} {p_b64_throughput:>10.0f}")

PRODUCTION PIPELINE BENCHMARK SUMMARY

Component                           Variant                   Metric                    Value
------------------------------------------------------------------------------------------
ViT-L/14                            Compiled, bs=128          Throughput (FPS)          918.5
                                                              Latency p50 (ms)         139.36
                                                              GPU mem (MB)               1939
Global MLP (CPU)                    Dynamic INT8              Batch FPS (b=32)         111393
                                                              Single FPS                11800
                                                              Model size (MB)            0.47
Personalized MLP (CPU)              Graph-optimized           Batch FPS (b=32)          81902
                                                              Single FPS                12522
                        

In [67]:
# runs in jupyter container on node-serve-model
# Production workload estimates
print("\n" + "=" * 80)
print("PRODUCTION WORKLOAD ESTIMATES")
print("=" * 80)

print(f"\n{'Workload':<45} {'Images':>10} {'Time':>12}")
print("-" * 70)

# Onboarding
onboard_imgs = 5000
onboard_vit = onboard_imgs / vit_fps
onboard_mlp = onboard_imgs / g_b64_throughput
print(f"{'Onboarding (1 user, 5K photos)':<45} {onboard_imgs:>10,} {onboard_vit + onboard_mlp:>10.1f}s")
print(f"{'  └ ViT encoding':<45} {'':<10} {onboard_vit:>10.1f}s")
print(f"{'  └ Global MLP scoring (Triton)':<45} {'':<10} {onboard_mlp:>10.2f}s")

# Daily new photos
daily_imgs = 5
daily_time = daily_imgs / (num_trials/global_triton_single.sum())
print(f"{'Daily new photos (1 user, 5 photos)':<45} {daily_imgs:>10} {daily_time*1000:>10.1f}ms")

# Weekly re-scoring
for n_users in [100, 1000]:
    total = n_users * 5000
    rescore_time = total / p_b64_throughput
    print(f"{'Weekly re-score (' + str(n_users) + ' users × 5K)':<45} {total:>10,} {rescore_time:>10.1f}s")


PRODUCTION WORKLOAD ESTIMATES

Workload                                          Images         Time
----------------------------------------------------------------------
Onboarding (1 user, 5K photos)                     5,000        5.6s
  └ ViT encoding                                                5.4s
  └ Global MLP scoring (Triton)                                0.13s
Daily new photos (1 user, 5 photos)                    5       11.8ms
Weekly re-score (100 users × 5K)                 500,000       13.9s
Weekly re-score (1000 users × 5K)              5,000,000      139.2s


------------------------------------------------------------------------

## Triton Production Configuration Reference

The production Triton configs used in this notebook:

**Global MLP** (`models_triton_production/flickr_global/config.pbtxt`):

    name: "flickr_global"
    platform: "onnxruntime_onnx"
    max_batch_size: 128
    instance_group: [{ count: 2, kind: KIND_GPU, gpus: [0] }]
    dynamic_batching: { preferred_batch_size: [4, 8, 16, 32, 64, 128], max_queue_delay_microseconds: 50 }

**Personalized MLP** (`models_triton_production/flickr_personalized/config.pbtxt`):

    name: "flickr_personalized"
    platform: "onnxruntime_onnx"
    max_batch_size: 128
    instance_group: [{ count: 2, kind: KIND_GPU, gpus: [0] }]
    dynamic_batching: { preferred_batch_size: [4, 8, 16, 32, 64, 128], max_queue_delay_microseconds: 50 }

**Key design decisions:** - **2 instances**: Handles concurrent onboarding + daily traffic without queue buildup - **Dynamic batching ON**: Groups small single-image requests from concurrent users into efficient GPU batches - **preferred_batch_size \[4–128\]**: Future-proofed for traffic growth; small sizes handle current bursty traffic, large sizes ready for batch jobs - **max_queue_delay 50 µs**: Low enough to not add perceptible latency for sparse arrivals - **Always-on server**: Users across time zones — no good maintenance window

------------------------------------------------------------------------

## Part 5: gRPC Benchmarks

Triton also exposes a gRPC endpoint on port 8001. gRPC uses binary serialization (protobuf) and HTTP/2, which typically gives lower latency and higher throughput than the HTTP/REST endpoint — especially for small payloads like our 768-dim embeddings.

We repeat the same benchmarks using the gRPC client to compare.

In [68]:
# runs in jupyter container on node-serve-model
import tritonclient.grpc as grpcclient

TRITON_GRPC_URL = "triton_server:8001"
grpc_client = grpcclient.InferenceServerClient(url=TRITON_GRPC_URL)

print(f"gRPC server live:  {grpc_client.is_server_live()}")
print(f"gRPC server ready: {grpc_client.is_server_ready()}")
print()

for model_name in ["flickr_global", "flickr_personalized"]:
    ready = grpc_client.is_model_ready(model_name)
    meta = grpc_client.get_model_metadata(model_name)
    print(f"Model '{model_name}': ready={ready}")

gRPC server live:  True
gRPC server ready: True

Model 'flickr_global': ready=True
Model 'flickr_personalized': ready=True


In [69]:
# runs in jupyter container on node-serve-model
print("Triton gRPC endpoint URLs:")
print(f"  gRPC:              {TRITON_GRPC_URL}")
print(f"  (binary protobuf over HTTP/2 — no REST URLs)")
print()
print("Equivalent HTTP endpoints for reference:")
print(f"  Health:            http://triton_server:8000/v2/health/ready")
for model_name in ["flickr_global", "flickr_personalized"]:
    print(f"  [{model_name}]")
    print(f"    Inference:       http://triton_server:8000/v2/models/{model_name}/infer  (HTTP)")
    print(f"    Inference:       triton_server:8001 → flickr_global.infer()  (gRPC)")

Triton gRPC endpoint URLs:
  gRPC:              triton_server:8001
  (binary protobuf over HTTP/2 — no REST URLs)

Equivalent HTTP endpoints for reference:
  Health:            http://triton_server:8000/v2/health/ready
  [flickr_global]
    Inference:       http://triton_server:8000/v2/models/flickr_global/infer  (HTTP)
    Inference:       triton_server:8001 → flickr_global.infer()  (gRPC)
  [flickr_personalized]
    Inference:       http://triton_server:8000/v2/models/flickr_personalized/infer  (HTTP)
    Inference:       triton_server:8001 → flickr_global.infer()  (gRPC)


In [70]:
# runs in jupyter container on node-serve-model
# gRPC helper functions
def grpc_infer_global(client, embeddings):
    inputs = [grpcclient.InferInput("input", embeddings.shape, "FP32")]
    inputs[0].set_data_from_numpy(embeddings)
    outputs = [grpcclient.InferRequestedOutput("output")]
    result = client.infer(model_name="flickr_global", inputs=inputs, outputs=outputs)
    return result.as_numpy("output")

def grpc_infer_personalized(client, embeddings, user_indices):
    inp_emb = grpcclient.InferInput("embedding", embeddings.shape, "FP32")
    inp_emb.set_data_from_numpy(embeddings)
    inp_idx = grpcclient.InferInput("user_idx", user_indices.shape, "INT64")
    inp_idx.set_data_from_numpy(user_indices)
    outputs = [grpcclient.InferRequestedOutput("output")]
    result = client.infer(model_name="flickr_personalized", inputs=[inp_emb, inp_idx], outputs=outputs)
    return result.as_numpy("output")

# Sanity check
scores = grpc_infer_global(grpc_client, single_emb)
print(f"gRPC Global single:  {scores.flatten()[0]:.4f}")
scores = grpc_infer_personalized(grpc_client, single_emb, single_user_idx)
print(f"gRPC Personal single: {scores.flatten()[0]:.4f}")

gRPC Global single:  0.6003
gRPC Personal single: 0.2831


### Global MLP — gRPC Benchmark

In [71]:
# runs in jupyter container on node-serve-model
# gRPC: Global single-sample latency
num_trials = 500

for _ in range(20):
    grpc_infer_global(grpc_client, single_emb)

grpc_global_single = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_global(grpc_client, single_emb)
    grpc_global_single.append(time.time() - start)

grpc_global_single = np.array(grpc_global_single)
print(f"Global MLP — gRPC Single Sample (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_global_single)*1000:.2f} ms")
print(f"  95th percentile: {np.percentile(grpc_global_single, 95)*1000:.2f} ms")
print(f"  99th percentile: {np.percentile(grpc_global_single, 99)*1000:.2f} ms")
print(f"  Throughput:      {num_trials / grpc_global_single.sum():.0f} infer/s")

Global MLP — gRPC Single Sample (n=500)
  Median latency:  1.02 ms
  95th percentile: 1.25 ms
  99th percentile: 1.43 ms
  Throughput:      946 infer/s


In [72]:
# runs in jupyter container on node-serve-model
# gRPC: Global batch=32
num_trials = 200

for _ in range(20):
    grpc_infer_global(grpc_client, batch_32)

grpc_global_b32 = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_global(grpc_client, batch_32)
    grpc_global_b32.append(time.time() - start)

grpc_global_b32 = np.array(grpc_global_b32)
grpc_g_b32_throughput = (num_trials * 32) / grpc_global_b32.sum()
print(f"Global MLP — gRPC Batch=32 (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_global_b32)*1000:.2f} ms")
print(f"  Throughput:      {grpc_g_b32_throughput:.0f} samples/s")

Global MLP — gRPC Batch=32 (n=200)
  Median latency:  1.62 ms
  Throughput:      19960 samples/s


In [73]:
# runs in jupyter container on node-serve-model
# gRPC: Global batch=64
num_trials = 200

for _ in range(20):
    grpc_infer_global(grpc_client, batch_64)

grpc_global_b64 = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_global(grpc_client, batch_64)
    grpc_global_b64.append(time.time() - start)

grpc_global_b64 = np.array(grpc_global_b64)
grpc_g_b64_throughput = (num_trials * 64) / grpc_global_b64.sum()
print(f"Global MLP — gRPC Batch=64 (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_global_b64)*1000:.2f} ms")
print(f"  Throughput:      {grpc_g_b64_throughput:.0f} samples/s")

Global MLP — gRPC Batch=64 (n=200)
  Median latency:  2.22 ms
  Throughput:      29501 samples/s


### Personalized MLP — gRPC Benchmark

In [74]:
# runs in jupyter container on node-serve-model
# gRPC: Personalized single-sample latency
num_trials = 500

for _ in range(20):
    grpc_infer_personalized(grpc_client, single_emb, single_user_idx)

grpc_personal_single = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_personalized(grpc_client, single_emb, single_user_idx)
    grpc_personal_single.append(time.time() - start)

grpc_personal_single = np.array(grpc_personal_single)
print(f"Personalized MLP — gRPC Single Sample (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_personal_single)*1000:.2f} ms")
print(f"  95th percentile: {np.percentile(grpc_personal_single, 95)*1000:.2f} ms")
print(f"  99th percentile: {np.percentile(grpc_personal_single, 99)*1000:.2f} ms")
print(f"  Throughput:      {num_trials / grpc_personal_single.sum():.0f} infer/s")

Personalized MLP — gRPC Single Sample (n=500)
  Median latency:  0.92 ms
  95th percentile: 0.96 ms
  99th percentile: 1.01 ms
  Throughput:      948 infer/s


In [75]:
# runs in jupyter container on node-serve-model
# gRPC: Personalized batch=32
num_trials = 200

for _ in range(20):
    grpc_infer_personalized(grpc_client, batch_32, batch_32_user_idx)

grpc_personal_b32 = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_personalized(grpc_client, batch_32, batch_32_user_idx)
    grpc_personal_b32.append(time.time() - start)

grpc_personal_b32 = np.array(grpc_personal_b32)
grpc_p_b32_throughput = (num_trials * 32) / grpc_personal_b32.sum()
print(f"Personalized MLP — gRPC Batch=32 (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_personal_b32)*1000:.2f} ms")
print(f"  Throughput:      {grpc_p_b32_throughput:.0f} samples/s")

Personalized MLP — gRPC Batch=32 (n=200)
  Median latency:  1.25 ms
  Throughput:      25487 samples/s


In [76]:
# runs in jupyter container on node-serve-model
# gRPC: Personalized batch=64
num_trials = 200

for _ in range(20):
    grpc_infer_personalized(grpc_client, batch_64, batch_64_user_idx)

grpc_personal_b64 = []
for _ in range(num_trials):
    start = time.time()
    grpc_infer_personalized(grpc_client, batch_64, batch_64_user_idx)
    grpc_personal_b64.append(time.time() - start)

grpc_personal_b64 = np.array(grpc_personal_b64)
grpc_p_b64_throughput = (num_trials * 64) / grpc_personal_b64.sum()
print(f"Personalized MLP — gRPC Batch=64 (n={num_trials})")
print(f"  Median latency:  {np.median(grpc_personal_b64)*1000:.2f} ms")
print(f"  Throughput:      {grpc_p_b64_throughput:.0f} samples/s")

Personalized MLP — gRPC Batch=64 (n=200)
  Median latency:  1.65 ms
  Throughput:      38201 samples/s


### HTTP vs gRPC Comparison

In [77]:
# runs in jupyter container on node-serve-model
print("=" * 80)
print("HTTP vs gRPC COMPARISON")
print("=" * 80)

print(f"\n{'Scenario':<40} {'HTTP (ms)':>10} {'gRPC (ms)':>10} {'Speedup':>10}")
print("-" * 70)

scenarios = [
    ("Global single", np.median(global_triton_single), np.median(grpc_global_single)),
    ("Global batch=32", np.median(global_triton_b32), np.median(grpc_global_b32)),
    ("Global batch=64", np.median(global_triton_b64), np.median(grpc_global_b64)),
    ("Personal single", np.median(personal_triton_single), np.median(grpc_personal_single)),
    ("Personal batch=32", np.median(personal_triton_b32), np.median(grpc_personal_b32)),
    ("Personal batch=64", np.median(personal_triton_b64), np.median(grpc_personal_b64)),
]

for name, http_ms, grpc_ms in scenarios:
    speedup = http_ms / grpc_ms if grpc_ms > 0 else float('inf')
    print(f"{name:<40} {http_ms*1000:>10.2f} {grpc_ms*1000:>10.2f} {speedup:>9.2f}x")

HTTP vs gRPC COMPARISON

Scenario                                  HTTP (ms)  gRPC (ms)    Speedup
----------------------------------------------------------------------
Global single                                  0.92       1.02      0.90x
Global batch=32                                1.30       1.62      0.81x
Global batch=64                                1.72       2.22      0.78x
Personal single                                0.84       0.92      0.92x
Personal batch=32                              1.24       1.25      1.00x
Personal batch=64                              1.78       1.65      1.08x


### `perf_analyzer` — gRPC Benchmarks

Run these on the **host** to compare gRPC vs HTTP with `perf_analyzer`:

#### Global MLP — gRPC concurrency sweep

``` bash
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_global -b 1 --shape input:768 --concurrency-range 1
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_global -b 1 --shape input:768 --concurrency-range 8
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_global -b 1 --shape input:768 --concurrency-range 16
```

#### Personalized MLP — gRPC concurrency sweep

``` bash
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 1
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 8
docker exec triton_production_sdk perf_analyzer -u localhost:8001 -i grpc -m flickr_personalized -b 1 --shape embedding:768 --shape user_idx:1 --concurrency-range 16
```

## Cleanup

When you are done, download the fully executed notebook for later reference.

Then, bring down the production stack:

``` bash
# runs on node-serve-model
docker compose -f ~/aesthetic-hub-serving/docker/docker-compose-production.yaml down
```